# Capitolo 18 – Ricorsivo e iterativo a confronto (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb)

Gli algoritmi del capitolo, ricorsivi e iterativi, a confronto con **conteggi** (chiamate, iterazioni, confronti, profondità della ricorsione) e con qualche tempo misurato: **SumExplicitStack** (la ricorsione simulata con una pila), **MergeSort** ricorsivo e **MergeSortBottomUp**, la potenza veloce ricorsiva e **PowerIt**, **Kadane**, i tre algoritmi di Fibonacci **FibRec**, **FibMemo** e **FibIt**, **GCDRec**/**GCDIt** e la ricerca binaria ricorsiva e iterativa (eliminazione della ricorsione in coda). I conteggi sono uguali in ogni linguaggio e in ogni macchina; i tempi (in ms) dipendono dalla macchina e dal linguaggio, contano solo come crescono.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/18-ricorsivo-iterativo/).

Ogni programma è scritto in un file `.java` con la cella `%%writefile` e poi compilato ed eseguito con `javac` e `java`: si eseguono le celle in ordine, dall'alto in basso (la prima cella verifica o installa il JDK). Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

In [1]:
# Verifica (ed eventualmente installa) il JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [2]:
%%writefile ExplicitStack.java
import java.util.*;

public class ExplicitStack {
    static int maxP = 0;   // massima dimensione della pila P

    // sumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita
    static long sumExplicitStack(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();   // pila dei sotto-array da sommare
        P.push(new int[]{0, n - 1});
        maxP = Math.max(maxP, P.size());
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];           // caso base
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                maxP = Math.max(maxP, P.size());
            }
        }
        return s;
    }

    // la cima della pila è a destra: si stampa dal fondo alla cima
    static String fmtP(ArrayDeque<int[]> P) {
        StringBuilder sb = new StringBuilder("[");
        Iterator<int[]> it = P.descendingIterator();
        boolean first = true;
        while (it.hasNext()) {
            int[] x = it.next();
            if (!first) sb.append(",");
            sb.append("(").append(x[0]).append(",").append(x[1]).append(")");
            first = false;
        }
        return sb.append("]").toString();
    }

    static long sumExplicitStackTrace(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();
        P.push(new int[]{0, n - 1});
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];
                System.out.println("pop (" + l + "," + r + "): caso base, s=" + s + " => P=" + fmtP(P));
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                System.out.println("pop (" + l + "," + r + "): mid=" + mid + ", push (" + (mid + 1) + "," + r + ") e (" + l + "," + mid + ") => P=" + fmtP(P) + ", s=" + s);
            }
        }
        return s;
    }

    // ---- le altre somme del codice del corso, con la profondità ----
    static int depth = 0;
    static int maxDepth = 0;

    static long sumIt(int[] a, int n) {
        long s = 0;
        for (int i = 0; i < n; i++) {
            s = s + a[i];
        }
        return s;
    }

    static long sumRecHalving(int[] a, int l, int r) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (l == r) {
            result = a[l];
        } else {
            int mid = l + (r - l) / 2;
            result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
        }
        depth--;
        return result;
    }

    static long sumRecDecrease(int[] a, int n) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (n == 0) {
            result = 0;
        } else {
            result = sumRecDecrease(a, n - 1) + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
    static int stepNo = 0;
    static int maxActive = 0;

    static void emit(ArrayList<String> stack, String action) {
        stepNo++;
        if (stack.size() > maxActive) maxActive = stack.size();
        System.out.printf("%2d | %s | %s%n", stepNo, String.join(" ", stack), action);
    }

    static long sumRecDecreaseTrace(int[] a, int n, ArrayList<String> stack) {
        stack.add("s(" + n + ")");
        if (n == 0) {
            emit(stack, "n=0: return 0");
            stack.remove(stack.size() - 1);
            return 0;
        }
        emit(stack, "n=" + n + " != 0: chiama s(" + (n - 1) + "), poi somma a[" + (n - 1) + "]");
        long r = sumRecDecreaseTrace(a, n - 1, stack);
        emit(stack, "return " + r + "+a[" + (n - 1) + "]=" + (r + a[n - 1]));
        stack.remove(stack.size() - 1);
        return r + a[n - 1];
    }

    public static void main(String[] args) {
        int[] a = {4, 7, 1, 9};
        int n = 4;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("SumExplicitStack(a, 4):");
        long res = sumExplicitStackTrace(a, n);
        maxP = 0;
        sumExplicitStack(a, n);
        System.out.println("risultato " + res + ", dimensione massima di P = " + maxP);
        System.out.println();
        System.out.println("sumIt                = " + sumIt(a, n));
        maxDepth = 0;
        long r1 = sumRecHalving(a, 0, n - 1);
        System.out.println("sumRecHalving        = " + r1 + "   profondità massima = " + maxDepth);
        maxDepth = 0;
        long r2 = sumRecDecrease(a, n);
        System.out.println("sumRecDecrease       = " + r2 + "   profondità massima = " + maxDepth);

        int[] b = {4, 7, 1};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   sumRecDecrease(a, 3)");
        System.out.println(" # | pila delle chiamate attive | azione");
        res = sumRecDecreaseTrace(b, 3, new ArrayList<>());
        System.out.println("risultato " + res + ", chiamate attive al massimo = " + maxActive + " = n+1");
    }
}

Writing ExplicitStack.java


In [3]:
!javac ExplicitStack.java && java ExplicitStack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [4]:
%%writefile ProvaExplicitStack.java
import java.util.*;

public class ProvaExplicitStack {
    public static void main(String[] args) {
        // Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
        int[] c = new int[100];
        for (int i = 0; i < 100; i++) c[i] = i + 1;
        ExplicitStack.maxP = 0;
        long s = ExplicitStack.sumExplicitStack(c, 100);
        System.out.println(s + " " + ExplicitStack.maxP);
    }
}

Writing ProvaExplicitStack.java


In [5]:
!javac ProvaExplicitStack.java && java ProvaExplicitStack

5050 8


## 2. MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [6]:
%%writefile BottomUp.java
import java.util.*;

public class BottomUp {
    static long comparisons = 0;   // confronti fra elementi

    // merge(a, l, m, r) del capitolo 17
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;
        int n2 = r - m;
        int[] L = new int[n1];
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            j++;
            k++;
        }
    }

    static void mergeSort(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSort(a, l, m);
            mergeSort(a, m + 1, r);
            merge(a, l, m, r);
        }
    }

    // mergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione
    static void mergeSortBottomUp(int[] a, int n) {
        int width = 1;
        while (width < n) {
            int l = 0;
            while (l + width < n) {                 // pseudocodice: l + width <= n
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                l = l + 2 * width;
            }
            width = 2 * width;
        }
    }

    static void mergeSortBottomUpTrace(int[] a, int n) {
        int width = 1;
        int passes = 0;
        while (width < n) {
            int l = 0;
            StringBuilder parts = new StringBuilder();
            while (l + width < n) {
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                if (parts.length() > 0) parts.append(", ");
                parts.append("a[" + l + ".." + m + "] con a[" + (m + 1) + ".." + r + "]");
                l = l + 2 * width;
            }
            passes++;
            System.out.println("width=" + width + ": merge " + parts + " => " + Arrays.toString(a));
            width = 2 * width;
        }
        System.out.println("passate = " + passes);
    }

    // insertionSort(a, n) (capitolo 3), con il conteggio dei confronti
    static void insertionSort(int[] a, int n) {
        for (int j = 1; j < n; j++) {
            int key = a[j];
            int i = j - 1;
            while (i >= 0) {
                comparisons++;
                if (a[i] > key) {
                    a[i + 1] = a[i];
                    i = i - 1;
                } else {
                    break;
                }
            }
            a[i + 1] = key;
        }
    }

    // Generatore congruenziale lineare: stessi numeri in Python, C++ e Java
    static int[] randomArray(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = 1 + (int) (x % 1000000L);
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {5, 2, 4, 6, 1, 3};
        System.out.println("a = " + Arrays.toString(a) + "   n = " + 6);
        mergeSortBottomUpTrace(a, 6);

        System.out.println();
        System.out.printf("%6s %14s %10s %9s  confronti%n", "n", "InsertionSort", "MergeSort", "BottomUp");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] base = randomArray(n, 12345);
            int[] x1 = base.clone(), x2 = base.clone(), x3 = base.clone();
            comparisons = 0;
            double t0 = nowMs(); insertionSort(x1, n); double t1 = nowMs() - t0;
            long c1 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSort(x2, 0, n - 1); double t2 = nowMs() - t0;
            long c2 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSortBottomUp(x3, n); double t3 = nowMs() - t0;
            long c3 = comparisons;
            int[] ord = base.clone();
            Arrays.sort(ord);
            String ok = (Arrays.equals(x1, x2) && Arrays.equals(x2, x3) && Arrays.equals(x3, ord)) ? "OK" : "ERROR";
            System.out.printf("%6d %14d %10d %9d  %s%n", n, c1, c2, c3, ok);
            times[idx++] = new double[]{n, t1, t2, t3};
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (double[] t : times) {
            System.out.printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms%n", (int) t[0], t[1], t[2], t[3]);
        }
    }
}

Writing BottomUp.java


In [7]:
!javac BottomUp.java && java BottomUp

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort      2.445 ms   MergeSort    0.589 ms   MergeSortBottomUp    0.165 ms
n=2000   InsertionSort      0.794 ms   MergeSort    0.192 ms   MergeSortBottomUp    0.261 ms
n=4000   InsertionSort      5.153 ms   MergeSort    0.407 ms   MergeSortBottomUp    0.610 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [8]:
%%writefile ProvaBottomUp.java
import java.util.*;

public class ProvaBottomUp {
    public static void main(String[] args) {
        // Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
        int n = 8000;
        int[] base = BottomUp.randomArray(n, 12345);
        int[] x1 = base.clone(), x2 = base.clone();
        BottomUp.comparisons = 0;
        BottomUp.insertionSort(x1, n);
        long c1 = BottomUp.comparisons;
        BottomUp.comparisons = 0;
        BottomUp.mergeSortBottomUp(x2, n);
        System.out.println(c1 + " " + BottomUp.comparisons + " " + (Arrays.equals(x1, x2) ? "True" : "False"));
    }
}

Writing ProvaBottomUp.java


In [9]:
!javac ProvaBottomUp.java && java ProvaBottomUp

16115652 93756 True


## 3. Potenza veloce: ricorsiva e PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — ricorsiva: $x^n=(x^{\lfloor n/2\rfloor})^2$ se $n$ è pari, $(x^{\lfloor n/2\rfloor})^2\cdot x$ se è dispari. $\Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor+2$ chiamate (profondità $\Theta(\log_2 n)$). Il risultato della chiamata ricorsiva è salvato in $y$ e usato due volte.
- **PowerIt**$(x,n)$ — *Input*: un numero $x$ e un naturale $n$. *Output*: $x^n$. Legge i bit di $n$ dal meno significativo: $\lfloor\log_2 n\rfloor+1$ iterazioni, memoria extra $\Theta(1)$; vale $p\cdot y^k=x^n$ a ogni iterazione.

Con $x=2$ e $n=13=(1101)_2$ si stampa la traccia del capitolo: $p$ viene moltiplicato per $x$, $x^4$ e $x^8$ (i bit di 13 uguali a 1) e $p=x^{13}$ dopo $4$ iterazioni. Poi la tabella: stesso risultato, $\lfloor\log_2 n\rfloor+1$ iterazioni contro $\lfloor\log_2 n\rfloor+2$ chiamate, contro le $n$ moltiplicazioni del ciclo naïf.

In [10]:
%%writefile Power.java
import java.util.*;

public class Power {
    static int powCalls = 0;   // chiamate di FastPower
    static int powIters = 0;   // iterazioni di PowerIt

    // fastPower(x, n): x^n per ricorsione sul dimezzamento di n
    static long fastPower(long x, int n) {
        powCalls++;
        if (n == 0) {
            return 1;                   // caso base
        }
        long y = fastPower(x, n / 2);   // una sola chiamata ricorsiva
        if (n % 2 == 0) {
            return y * y;               // n pari
        }
        return x * y * y;               // n dispari
    }

    // powerIt(x, n): x^n con square-and-multiply
    static long powerIt(long x, int n) {
        long p = 1;
        long y = x;
        int k = n;
        while (k > 0) {
            powIters++;
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            k = k / 2;
        }
        return p;
    }

    static void powerItTrace(long x, int n) {
        System.out.println("PowerIt(" + x + ", " + n + "): " + n + " = (" + Integer.toBinaryString(n) + ")_2");
        long p = 1, y = x;
        int k = n, it = 0;
        while (k > 0) {
            it++;
            String par = k % 2 == 1 ? "dispari" : "pari";
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            System.out.println(it + ". k=" + k + " " + par + ": p=" + p + ", y=" + y + ", k=" + (k / 2));
            k = k / 2;
        }
        System.out.println("iterazioni = " + it + ", risultato p = " + p);
    }

    public static void main(String[] args) {
        powerItTrace(2, 13);
        System.out.println();
        System.out.printf("%3s %12s %8s %10s %6s   (iterazioni, chiamate, moltiplicazioni)%n", "n", "2^n", "PowerIt", "FastPower", "naive");
        int[] ns = {0, 1, 2, 5, 13, 30};
        for (int n : ns) {
            powIters = 0;
            long r1 = powerIt(2, n);
            powCalls = 0;
            long r2 = fastPower(2, n);
            System.out.printf("%3d %12d %8d %10d %6d   %s%n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1L << n)) ? "OK" : "ERROR");
        }
    }
}

Writing Power.java


In [11]:
!javac Power.java && java Power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 dispari: p=2, y=4, k=6
2. k=6 pari: p=2, y=16, k=3
3. k=3 dispari: p=32, y=256, k=1
4. k=1 dispari: p=8192, y=65536, k=0
iterazioni = 4, risultato p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterazioni, chiamate, moltiplicazioni)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Prova tu.** Prova un'altra base e un altro esponente (in C++ e Java il risultato deve stare in 64 bit).

In [12]:
%%writefile ProvaPower.java
import java.util.*;

public class ProvaPower {
    public static void main(String[] args) {
        // Prova un'altra base e un altro esponente (attenzione: in C++ e Java il risultato deve stare in 64 bit).
        Power.powIters = 0;
        Power.powCalls = 0;
        long r1 = Power.powerIt(3, 25);
        long r2 = Power.fastPower(3, 25);
        System.out.println(r1 + " " + Power.powIters + " " + r2 + " " + Power.powCalls);
    }
}

Writing ProvaPower.java


In [13]:
!javac ProvaPower.java && java ProvaPower

847288609443 5 847288609443 6


## 4. Kadane: un algoritmo iterativo migliore

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma massima di un sottoarray non vuoto $a[l..r]$. È la versione compatta del capitolo: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. Il ciclo fa $n-1$ iterazioni: $\Theta(n)$ passi e $\Theta(1)$ memoria, contro le $n(n+1)/2$ coppie $(l,r)$ della forza bruta ($\Theta(n^2)$) e contro il divide et impera ricorsivo ($\Theta(n\log_2 n)$): un buon algoritmo iterativo può battere la ricorsione.

Sull'esempio delle variazioni di prezzo del capitolo 17 il risultato è $43$. Poi, su array pseudo-casuali con valori fra $-50$ e $50$ (stesso generatore in tutti i linguaggi, seme $n$), le iterazioni di `Kadane`, le coppie esaminate dalla forza bruta e la verifica che i due algoritmi trovino la stessa somma massima; infine i tempi misurati.

In [14]:
%%writefile KadaneCompact.java
import java.util.*;

public class KadaneCompact {
    // kadane(a, n): somma massima di un sottoarray non vuoto di a[0..n-1]
    static int kadane(int[] a, int n) {
        int best = a[0];
        int cur = a[0];                 // somma massima di un sottoarray che termina in i
        int i = 1;
        while (i < n) {
            cur = Math.max(a[i], cur + a[i]);
            best = Math.max(best, cur);
            i = i + 1;
        }
        return best;
    }

    // Forza bruta: restituisce {somma massima, coppie (l, r) esaminate}
    static long[] bruteForceBest(int[] a, int n) {
        int best = a[0];
        long pairs = 0;
        for (int l = 0; l < n; l++) {
            int s = 0;
            for (int r = l; r < n; r++) {
                s = s + a[r];
                pairs++;
                if (s > best) best = s;
            }
        }
        return new long[]{best, pairs};
    }

    // Interi fra -50 e 50 (generatore congruenziale lineare)
    static int[] randomValues(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = (int) (x % 101L) - 50;
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        System.out.println("a = " + Arrays.toString(a));
        System.out.println("Kadane(a, 16) = " + kadane(a, 16));
        int[] b = {-3, -1, -2, -5};
        System.out.println("a = " + Arrays.toString(b) + "  Kadane(a, 4) = " + kadane(b, 4));
        System.out.println();
        System.out.printf("%6s %7s %11s %11s %9s  stessa somma?%n", "n", "Kadane", "BruteForce", "iterations", "pairs");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] v = randomValues(n, n);
            double t0 = nowMs(); int k = kadane(v, n); double t1 = nowMs() - t0;
            t0 = nowMs(); long[] bf = bruteForceBest(v, n); double t2 = nowMs() - t0;
            System.out.printf("%6d %7d %11d %11d %9d  %s%n", n, k, bf[0], n - 1, bf[1], k == bf[0] ? "OK" : "ERROR");
            times[idx++] = new double[]{n, t1, t2};
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (double[] t : times) {
            System.out.printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms%n", (int) t[0], t[1], t[2]);
        }
    }
}

Writing KadaneCompact.java


In [15]:
!javac KadaneCompact.java && java KadaneCompact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  stessa somma?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Tempi misurati (dipendono dalla macchina):
n=1000   Kadane    0.0499 ms   BruteForce      2.043 ms
n=2000   Kadane    0.1101 ms   BruteForce      5.252 ms
n=4000   Kadane    0.1928 ms   BruteForce      2.061 ms


**Prova tu.** Cambia $n$: le coppie della forza bruta crescono come $n^2$, le iterazioni di Kadane come $n$.

In [16]:
%%writefile ProvaKadaneCompact.java
import java.util.*;

public class ProvaKadaneCompact {
    public static void main(String[] args) {
        // Prova con n = 8000: le coppie della forza bruta sono n(n+1)/2 = 32 004 000, le iterazioni di Kadane 7999.
        int n = 8000;
        int[] v = KadaneCompact.randomValues(n, 12345);
        long[] bf = KadaneCompact.bruteForceBest(v, n);
        System.out.println(KadaneCompact.kadane(v, n) + " (" + bf[0] + ", " + bf[1] + ")");
    }
}

Writing ProvaKadaneCompact.java


In [17]:
!javac ProvaKadaneCompact.java && java ProvaKadaneCompact

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [18]:
%%writefile Fibonacci.java
import java.util.*;

public class Fibonacci {
    static long fibCalls = 0;   // chiamate eseguite

    // fibRec(n)
    static long fibRec(int n) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        return fibRec(n - 1) + fibRec(n - 2);
    }

    // fibMemo(n, memo): memo[k] = -1 significa «non definito»
    static long fibMemo(int n, long[] memo) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        if (memo[n] == -1) {
            memo[n] = fibMemo(n - 1, memo) + fibMemo(n - 2, memo);
        }
        return memo[n];
    }

    // fibIt(n)
    static long fibIt(int n) {
        if (n <= 1) {
            return n;
        }
        long prev = 0;                  // F(i-2)
        long curr = 1;                  // F(i-1)
        int i = 2;
        while (i <= n) {
            long next = prev + curr;
            prev = curr;
            curr = next;
            i = i + 1;
        }
        return curr;
    }

    // Come fibRec, ma conta le chiamate per ogni valore del parametro
    static long fibRecHist(int n, int[] counts) {
        counts[n]++;
        if (n <= 1) {
            return n;
        }
        return fibRecHist(n - 1, counts) + fibRecHist(n - 2, counts);
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] counts = new int[6];
        long f5 = fibRecHist(5, counts);
        System.out.println("FibRec(5) = " + f5);
        StringBuilder line = new StringBuilder();
        int total = 0;
        for (int k = 5; k >= 0; k--) {
            if (k < 5) line.append("  ");
            line.append(k).append(":").append(counts[k]);
            total += counts[k];
        }
        System.out.println("chiamate per ogni valore del parametro: " + line + "   totale " + total);
        System.out.println();
        System.out.printf("%3s %8s %9s %10s %8s %6s  (chiamate, chiamate, chiamate, iterazioni)%n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
        int[] ns = {10, 15, 20, 25, 30};
        for (int n : ns) {
            long f = fibIt(n);
            fibCalls = 0;
            long r1 = fibRec(n);
            long c1 = fibCalls;
            fibCalls = 0;
            long[] memo = new long[n + 1];
            Arrays.fill(memo, -1);
            long r2 = fibMemo(n, memo);
            long c2 = fibCalls;
            boolean ok = (r1 == r2 && r2 == f && c1 == 2 * fibIt(n + 1) - 1 && c2 == 2 * n - 1);
            System.out.printf("%3d %8d %9d %10d %8d %6d  %s%n", n, f, c1, 2 * fibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (int n : new int[]{20, 25, 30}) {
            double t0 = nowMs(); fibRec(n); double t1 = nowMs() - t0;
            t0 = nowMs();
            long sink = 0;
            for (int rep = 0; rep < 1000; rep++) {
                sink += fibIt(n);
            }
            double t2 = (nowMs() - t0) / 1000;
            System.out.printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms%n", n, t1, t2);
        }
    }
}

Writing Fibonacci.java


In [19]:
!javac Fibonacci.java && java Fibonacci

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     0.0294 ms   FibIt   0.000118 ms
n=25  FibRec     0.3402 ms   FibIt   0.000059 ms
n=30  FibRec     3.3858 ms   FibIt   0.000034 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [20]:
%%writefile ProvaFibonacci.java
import java.util.*;

public class ProvaFibonacci {
    public static void main(String[] args) {
        // Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
        int n = 32;
        Fibonacci.fibCalls = 0;
        long r1 = Fibonacci.fibRec(n);
        System.out.println(r1 + " " + Fibonacci.fibCalls);
        Fibonacci.fibCalls = 0;
        long[] memo = new long[n + 1];
        Arrays.fill(memo, -1);
        long r2 = Fibonacci.fibMemo(n, memo);
        System.out.println(r2 + " " + Fibonacci.fibCalls);
    }
}

Writing ProvaFibonacci.java


In [21]:
!javac ProvaFibonacci.java && java ProvaFibonacci

2178309 7049155
2178309 63


## 6. Ricorsione in coda e la sua eliminazione: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

Una chiamata ricorsiva è *in coda* se è l'ultima operazione e il suo risultato è restituito così com'è: si può sostituire con l'aggiornamento dei parametri e un salto all'inizio, cioè con un ciclo (condizione del ciclo = negazione del caso base; chiamata in coda = assegnamento simultaneo dei parametri). Il risultato è lo stesso, il numero di passi ha lo stesso ordine di grandezza e la memoria extra scende da $\Theta(\text{profondità})$ a $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: due naturali $a$, $b$. *Output*: il loro massimo comun divisore (algoritmo di Euclide). Con $(a,b)=(252,105)$ entrambi passano per le coppie $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ e restituiscono $21$: quattro chiamate contro tre iterazioni. La variabile temporanea $r$ serve perché $a\bmod b$ va calcolato con il *vecchio* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: un array ordinato, due indici $l$, $r$ e un valore $m$. *Output*: un indice di $m$ in $a[l..r]$ oppure $-1$. `ItBS(a, 1, n, m)` è la ricerca binaria iterativa del capitolo «Ricerca».

In [22]:
%%writefile TailRecursion.java
import java.util.*;

public class TailRecursion {
    static ArrayList<String> gcdLog = new ArrayList<>();   // coppie (a, b) visitate (solo per la stampa)
    static int rbsCalls = 0;   // chiamate di RecBS
    static int bsIters = 0;    // iterazioni di ItBS

    static String pairStr(long a, long b) {
        return "(" + a + "," + b + ")";
    }

    // gcdRec(a, b)
    static long gcdRec(long a, long b) {
        gcdLog.add(pairStr(a, b));
        if (b == 0) {
            return a;
        }
        return gcdRec(b, a % b);            // chiamata in coda
    }

    // gcdIt(a, b)
    static long gcdIt(long a, long b) {
        gcdLog.add(pairStr(a, b));
        while (b != 0) {                    // negazione del caso base
            long r = a % b;                 // con il vecchio a
            a = b;
            b = r;
            gcdLog.add(pairStr(a, b));
        }
        return a;
    }

    // recBS(a, l, r, m)
    static int recBS(int[] a, int l, int r, int m) {
        rbsCalls++;
        if (l > r) {
            return -1;
        }
        int mid = l + (r - l) / 2;
        if (a[mid] == m) {
            return mid;
        }
        if (a[mid] < m) {
            return recBS(a, mid + 1, r, m);     // chiamata in coda
        } else {
            return recBS(a, l, mid - 1, m);     // chiamata in coda
        }
    }

    // itBS(a, l, r, m)
    static int itBS(int[] a, int l, int r, int m) {
        while (l <= r) {                        // negazione del caso base l > r
            bsIters++;
            int mid = l + (r - l) / 2;
            if (a[mid] == m) {
                return mid;                     // il secondo caso base resta nel ciclo
            }
            if (a[mid] < m) {
                l = mid + 1;
            } else {
                r = mid - 1;
            }
        }
        return -1;
    }

    public static void main(String[] args) {
        long[][] pairsAB = {{252, 105}, {1071, 462}, {832040, 514229}};
        for (long[] p : pairsAB) {
            long a = p[0], b = p[1];
            gcdLog.clear();
            long g1 = gcdRec(a, b);
            int calls = gcdLog.size();
            String pairs = String.join(" ", gcdLog);
            gcdLog.clear();
            long g2 = gcdIt(a, b);
            int iters = gcdLog.size() - 1;
            System.out.printf("GCDRec(%d, %d) = %d   chiamate=%d     GCDIt = %d   iterazioni=%d   %s%n", a, b, g1, calls, g2, iters, g1 == g2 ? "OK" : "ERROR");
            if (a == 252) {
                System.out.println("  coppie: " + pairs);
            }
        }

        System.out.println();
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.printf("%4s %6s %6s %6s %11s%n", "m", "RecBS", "calls", "ItBS", "iterations");
        int[] ms = {12, 10, 3, 19, 100};
        for (int m : ms) {
            rbsCalls = 0;
            int r1 = recBS(a, 0, n - 1, m);
            bsIters = 0;
            int r2 = itBS(a, 0, n - 1, m);
            System.out.printf("%4d %6d %6d %6d %11d   %s%n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
        }
    }
}

Writing TailRecursion.java


In [23]:
!javac TailRecursion.java && java TailRecursion

GCDRec(252, 105) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
  coppie: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
GCDRec(832040, 514229) = 1   chiamate=29     GCDIt = 1   iterazioni=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Prova tu.** Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.

In [24]:
%%writefile ProvaTailRecursion.java
import java.util.*;

public class ProvaTailRecursion {
    public static void main(String[] args) {
        // Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.
        TailRecursion.gcdLog.clear();
        long g = TailRecursion.gcdRec(46368, 28657);
        System.out.println(g + " " + TailRecursion.gcdLog.size());
    }
}

Writing ProvaTailRecursion.java


In [25]:
!javac ProvaTailRecursion.java && java ProvaTailRecursion

1 23


## 7. Profondità della ricorsione e pila delle chiamate

La memoria usata dalla pila delle chiamate è proporzionale alla **profondità** $D(n)$ della ricorsione (il massimo numero di chiamate attive contemporaneamente), non al numero di chiamate. La tabella confronta la profondità **misurata** con la formula del capitolo:

| algoritmo | $D(n)$ |
|---|---|
| somma per decremento di uno (caso base $n=0$) | $n+1$ |
| somma per dimezzamento | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (caso peggiore, valore assente) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

Una profondità $\Theta(\log_2 n)$ non è mai un problema in pratica; una profondità $\Theta(n)$ rischia lo **stack overflow**: l'ultima parte della cella (le righe che iniziano con `~~`) dipende dal linguaggio e dalla macchina.

In [26]:
%%writefile Depth.java
import java.util.*;

public class Depth {
    static int depth = 0;
    static int maxDepth = 0;
    static int calls = 0;

    static void enter() {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
    }

    static long sumRecDecrease(int[] a, int n) {
        enter();
        long result = (n == 0) ? 0 : sumRecDecrease(a, n - 1) + a[n - 1];
        depth--;
        return result;
    }

    static long sumRecHalving(int[] a, int l, int r) {
        enter();
        long result;
        if (l == r) {
            result = a[l];
        } else {
            int mid = l + (r - l) / 2;
            result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
        }
        depth--;
        return result;
    }

    static long fastPower(long x, int n) {
        enter();
        long result;
        if (n == 0) {
            result = 1;
        } else {
            long y = fastPower(x, n / 2);
            result = (n % 2 == 0) ? y * y : x * y * y;
        }
        depth--;
        return result;
    }

    static int recBS(int[] a, int l, int r, int m) {
        calls++;
        if (l > r) {
            return -1;
        }
        int mid = l + (r - l) / 2;
        if (a[mid] == m) {
            return mid;
        }
        if (a[mid] < m) {
            return recBS(a, mid + 1, r, m);
        }
        return recBS(a, l, mid - 1, m);
    }

    static long fibRec(int n) {
        enter();
        long result = (n <= 1) ? n : fibRec(n - 1) + fibRec(n - 2);
        depth--;
        return result;
    }

    static int floorLog2(int n) {
        int f = 0;
        while ((1 << (f + 1)) <= n) f++;
        return f;
    }

    static int ceilLog2(int n) {
        int c = 0;
        while ((1 << c) < n) c++;
        return c;
    }

    public static void main(String[] args) {
        System.out.printf("%5s %9s %5s %8s %8s %10s %8s %6s %6s%n", "n", "decrease", "n+1", "halving", "formula", "FastPower", "formula", "RecBS", "bound");
        int[] ns = {10, 100, 500};
        for (int n : ns) {
            int[] a = new int[n];
            for (int i = 0; i < n; i++) a[i] = i;
            maxDepth = 0; sumRecDecrease(a, n); int d1 = maxDepth;
            maxDepth = 0; sumRecHalving(a, 0, n - 1); int d2 = maxDepth;
            maxDepth = 0; fastPower(1, n); int d3 = maxDepth;
            calls = 0; recBS(a, 0, n - 1, n + 100); int d4 = calls;
            System.out.printf("%5d %9d %5d %8d %8d %10d %8d %6d %6d%n", n, d1, n + 1, d2, ceilLog2(n) + 1, d3, floorLog2(n) + 2, d4, floorLog2(n) + 2);
        }
        System.out.println();
        System.out.printf("%3s %13s%n", "n", "FibRec depth");
        int[] fs = {5, 10, 20};
        for (int n : fs) {
            maxDepth = 0;
            fibRec(n);
            System.out.printf("%3d %13d%n", n, maxDepth);
        }

        try {
            depth = 0;
            sumRecDecrease(new int[1000000], 1000000);
            System.out.println("~~ Java: n = 1000000 OK");
        } catch (StackOverflowError e) {
            depth = 0;
            System.out.println("~~ Java: StackOverflowError for sumRecDecrease with n = 1000000 (depth n+1)");
        }
    }
}

Writing Depth.java


In [27]:
!javac Depth.java && java Depth

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ Java: StackOverflowError for sumRecDecrease with n = 1000000 (depth n+1)


**Prova tu.** Misura la profondità per un altro $n$ (qui $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [28]:
%%writefile ProvaDepth.java
import java.util.*;

public class ProvaDepth {
    public static void main(String[] args) {
        // Misura la profondità della ricorsione per un altro n (qui 2000): decrease = n+1, halving = ceil(log2 n) + 1.
        int n = 2000;
        int[] a = new int[n];
        Arrays.fill(a, 1);
        Depth.maxDepth = 0;
        Depth.sumRecDecrease(a, n);
        System.out.print(Depth.maxDepth + " ");
        Depth.maxDepth = 0;
        Depth.sumRecHalving(a, 0, n - 1);
        System.out.println(Depth.maxDepth);
    }
}

Writing ProvaDepth.java


In [29]:
!javac ProvaDepth.java && java ProvaDepth

2001 12
